# Chapter 02 데모 — AgentCore Runtime: 고객지원 에이전트를 서버리스로 배포하기

> 📑 대응 강의: **02-Runtime.pptx** (AgentCore Runtime 및 프레임워크 통합)

## 데모 시나리오: *AnyCompany Electronics 고객지원 에이전트*

전자제품 쇼핑몰의 고객지원 에이전트를 **챕터마다 한 단계씩 발전**시켜, 마지막 챕터에서
"인증된 고객을 기억하고, 사내 도구를 안전하게 쓰며, 품질이 측정되는 프로덕션 에이전트"를 완성합니다.

| 챕터 | 노트북 | 고객지원 에이전트에 추가되는 기능 |
|---|---|---|
| 👉 **02** | `02_Runtime_Demo` | Strands 에이전트 → **AgentCore Runtime** 서버리스 배포, 세션 격리, 버전/엔드포인트 |
| 03 | `03_SecurityIdentity_Demo` | **Cognito JWT 인바운드 인증**, 사용자 식별·감사 로그, **API 키 아웃바운드 인증**(배송 파트너 API) |
| 04 | `04_ToolsGateway_Demo` | **Code Interpreter**, MCP, **AgentCore Gateway**(Lambda→MCP 도구), **AgentCore Policy**(Cedar) |
| 05 | `05_Memory_Demo` | **AgentCore Memory** 단기/장기 메모리 → 세션을 넘어 고객을 기억 |
| 06 | `06_Observability_Evaluation_Demo` | **Observability**(트레이스/스팬/지표), **Evaluations**(온디맨드/온라인), PROD 승격 |
| 99 | `99_Cleanup` | 데모 리소스 일괄 삭제 |

### 이번 챕터에서 할 일
1. Strands Agents 로 고객지원 에이전트 프로토타입 만들기 (슬라이드 6)
2. `BedrockAgentCoreApp` 으로 감싸 Runtime 호환 에이전트로 만들기 — **코드 4줄** (슬라이드 23)
3. 로컬에서 `/invocations`, `/ping` 계약 확인 + 비동기 장기 실행 작업 (슬라이드 18, 25)
4. 스타터 툴킷으로 AgentCore Runtime 에 배포 (슬라이드 21–26)
5. **세션 격리**(microVM)와 세션 수명 주기 확인 (슬라이드 16–17)
6. **버전과 엔드포인트**: `PROD` 엔드포인트 만들기 (슬라이드 13–15)

```
[노트북] ──SigV4(IAM)──▶ AgentCore Runtime (세션별 microVM)
                             └─ Strands Agent ── Bedrock 모델
                                 ├─ get_product_info (로컬 도구)
                                 ├─ get_return_policy (로컬 도구)
                                 └─ generate_sales_report (비동기 작업)
```

### 사전 요구 사항
- Python 3.10+ / AWS 자격 증명 설정 (`aws configure`) / AgentCore 지원 리전 (예: `us-east-1`, `us-west-2`)
- Bedrock 모델 액세스: 기본값 **Claude Haiku 4.5** (`demo_helpers/config.py` 의 `MODEL_ID` 로 변경 가능)
- 배포는 **AWS CodeBuild** 에서 ARM64 컨테이너로 빌드되므로 로컬 Docker 는 필요하지 않습니다.

## 0. 환경 준비

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json, os, sys, time, subprocess
from pathlib import Path

import boto3
from IPython.display import Markdown, display

from demo_helpers import config
from demo_helpers.aws_setup import get_region, get_account_id, wait_for

REGION = get_region()
ACCOUNT_ID = get_account_id()
print(f"Region : {REGION}")
print(f"Account: {ACCOUNT_ID}")
print(f"Model  : {config.MODEL_ID}")

## 1. Strands Agents 로 프로토타입 만들기 (슬라이드 4–7)

에이전트를 만들려면 **모델 · 도구 · 시스템 프롬프트** 세 가지를 정의합니다.
도구는 `@tool` 데코레이터와 docstring 만으로 정의되며, 모델은 docstring 과 타입 힌트를 보고 도구 사용법을 이해합니다.

> 로컬 도구 코드는 [`demo_helpers/tools.py`](demo_helpers/tools.py) 에 있습니다. 런타임 컨테이너에도 그대로 함께 배포됩니다.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from demo_helpers.tools import get_product_info, get_return_policy, SYSTEM_PROMPT

print(SYSTEM_PROMPT)

model = BedrockModel(model_id=config.MODEL_ID, region_name=REGION, temperature=0.3)
local_agent = Agent(model=model, tools=[get_product_info, get_return_policy], system_prompt=SYSTEM_PROMPT)

In [ ]:
result = local_agent("AnySound 헤드폰 가격이랑 반품 조건 알려주세요.")

✅ 프로토타입이 동작합니다. 하지만 이 에이전트는 **내 노트북 프로세스 안에서만** 실행됩니다.
- 여러 고객이 동시에 쓰면? 고객 간 대화 상태가 섞이지 않게 하려면? 확장은?
- → **AgentCore Runtime** 이 이 문제들을 해결합니다.

## 2. Runtime 호환 에이전트로 변환 — 코드 4줄 (슬라이드 23)

`#### AGENTCORE RUNTIME ①~④ ####` 표시가 추가된 부분입니다.

| 표시 | 역할 |
|---|---|
| ① `from bedrock_agentcore.runtime import BedrockAgentCoreApp` | SDK 가져오기 |
| ② `app = BedrockAgentCoreApp()` | HTTP 서버(포트 8080) + `/invocations`, `/ping` 자동 구현 |
| ③ `@app.entrypoint` | 요청 페이로드를 받아 에이전트를 실행할 함수 지정 |
| ④ `app.run()` | 서버 실행 |

추가로 두 가지를 관찰용으로 넣었습니다.
- `MICROVM_BOOT_ID` : 프로세스(=microVM)가 시작될 때 한 번 생성되는 ID → **세션마다 다른 microVM** 인지 확인
- `generate_sales_report` : `app.add_async_task()` 로 **장기 실행 백그라운드 작업** 표시 (슬라이드 18)

In [ ]:
%%writefile agent/runtime_ch02.py
"""[Chapter 02] AgentCore Runtime 에서 실행되는 고객지원 에이전트 (v1)."""
import os
import threading
import time
import uuid

from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp  #### AGENTCORE RUNTIME ① ####

from demo_helpers.config import MODEL_ID
from demo_helpers.tools import SYSTEM_PROMPT, get_product_info, get_return_policy

REGION = os.environ.get("AWS_REGION", "us-east-1")
MICROVM_BOOT_ID = uuid.uuid4().hex[:8]  # 이 프로세스(microVM)가 시작될 때 1회 생성

app = BedrockAgentCoreApp()  #### AGENTCORE RUNTIME ② ####


@tool
def generate_sales_report(duration_seconds: int = 20) -> str:
    """장시간 걸리는 '판매 리포트 생성' 작업을 백그라운드로 시작하고 즉시 응답합니다.

    Args:
        duration_seconds: 작업 소요 시간(초)
    """
    task_id = app.add_async_task("sales_report", {"duration": duration_seconds})  # /ping → HealthyBusy

    def background_work():
        time.sleep(duration_seconds)  # 실제로는 데이터 집계/리포트 생성 로직
        app.complete_async_task(task_id)  # /ping → Healthy

    threading.Thread(target=background_work, daemon=True).start()
    return f"리포트 작업(ID: {task_id})을 시작했습니다. 약 {duration_seconds}초 후 완료됩니다."


# 세션 내 대화 상태: 같은 세션의 요청은 같은 microVM 으로 라우팅되므로 전역 객체에 유지됩니다.
_agent = None


def get_agent() -> Agent:
    global _agent
    if _agent is None:
        _agent = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.3),
            tools=[get_product_info, get_return_policy, generate_sales_report],
            system_prompt=SYSTEM_PROMPT,
            callback_handler=None,
        )
    return _agent


@app.entrypoint  #### AGENTCORE RUNTIME ③ ####
def invoke(payload, context):
    agent = get_agent()
    result = agent(payload.get("prompt", ""))
    return {
        "response": str(result),
        "session_id": context.session_id,
        "microvm_boot_id": MICROVM_BOOT_ID,
        "messages_in_this_microvm": len(agent.messages),
    }


if __name__ == "__main__":
    app.run()  #### AGENTCORE RUNTIME ④ ####

## 3. 로컬에서 실행해 보기 — `/invocations` · `/ping` 계약 (슬라이드 25)

AgentCore Runtime 이 컨테이너에 요구하는 계약은 단 두 개입니다.
- `POST /invocations` : 에이전트 호출
- `GET /ping` : 상태 확인 (`Healthy` / `HealthyBusy`)

에이전트 서버를 백그라운드 프로세스로 띄워 직접 호출해 봅니다.

In [ ]:
import requests

log_file = open("local_agent.log", "w", encoding="utf-8")
local_server = subprocess.Popen(
    [sys.executable, "-m", "agent.runtime_ch02"],
    stdout=log_file, stderr=subprocess.STDOUT,
    env={**os.environ, "AWS_REGION": REGION, "PYTHONIOENCODING": "utf-8"},
)

LOCAL = "http://127.0.0.1:8080"
for _ in range(30):
    try:
        print("GET /ping →", requests.get(f"{LOCAL}/ping", timeout=2).json())
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print(Path("local_agent.log").read_text(encoding="utf-8")[-3000:])
    raise RuntimeError("로컬 에이전트 서버가 시작되지 않았습니다.")

In [ ]:
def local_invoke(prompt, session_id):
    r = requests.post(
        f"{LOCAL}/invocations",
        headers={"X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id},
        json={"prompt": prompt},
        timeout=120,
    )
    return r.json()

r = local_invoke("안녕하세요, 제 이름은 김민수예요. 노트북 반품 정책 알려주세요.", "local-session-A")
display(Markdown(r["response"]))
print({k: v for k, v in r.items() if k != "response"})

### ⚠️ 세션 격리가 없으면? (슬라이드 16)
로컬 서버는 **하나의 프로세스**이므로 다른 세션(`local-session-B`)의 요청도 같은 메모리 상태를 공유합니다.
다른 고객이 앞 고객의 이름을 알아낼 수 있는지 확인해 보세요.

In [ ]:
r = local_invoke("제 이름이 뭐였죠?", "local-session-B")   # 다른 세션!
display(Markdown(r["response"]))
print({k: v for k, v in r.items() if k != "response"})

👉 같은 `microvm_boot_id` 에서 실행되어 **다른 세션의 대화 내용이 노출**됩니다. 배포 후에는 어떻게 달라지는지 5단계에서 비교합니다.

### 비동기 장기 실행 작업 (슬라이드 18)
`add_async_task()` 로 등록된 작업이 실행 중이면 `/ping` 이 `HealthyBusy` 를 반환합니다.
Runtime 은 이 상태를 보고 **작업 중인 세션을 유휴 상태로 판단해 종료하지 않습니다** (최대 8시간).

In [ ]:
r = local_invoke("판매 리포트 생성 작업을 15초짜리로 시작해 주세요.", "local-session-C")
display(Markdown(r["response"]))

for i in range(5):
    print(f"t+{i*5:>2}s  GET /ping →", requests.get(f"{LOCAL}/ping").json()["status"])
    time.sleep(5)

In [ ]:
local_server.terminate()
local_server.wait(timeout=10)
log_file.close()
print("로컬 서버 종료")

## 4. AgentCore Runtime 에 배포하기 (슬라이드 21–26)

### 4-1. 실행 역할(IAM) 준비
런타임이 에이전트를 실행할 때 사용하는 IAM 역할입니다. 이후 챕터에서 필요한 권한(Identity, Code Interpreter, Memory)까지
미리 포함해 두었습니다. 내용은 **03장(실행 역할 슬라이드)** 에서 자세히 살펴봅니다.

In [ ]:
from demo_helpers.aws_setup import ensure_runtime_execution_role

execution_role_arn = ensure_runtime_execution_role()
print(execution_role_arn)

### 4-2. 스타터 툴킷으로 구성(configure) → 배포(launch)

| 스타터 툴킷이 자동화하는 일 | 툴킷 없이 직접 한다면 |
|---|---|
| Dockerfile 생성, ECR 리포지토리 생성 | Dockerfile 작성, `docker build/push` |
| CodeBuild 로 ARM64 이미지 빌드 | ARM64 빌드 환경 준비 |
| `CreateAgentRuntime` / `UpdateAgentRuntime` 호출 | boto3 `bedrock-agentcore-control` 직접 호출 |
| 관찰성(OpenTelemetry) 계측 자동 적용 | `opentelemetry-instrument` 로 실행 |

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint="agent/runtime_ch02.py",
    agent_name=config.AGENT_NAME,
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="agent/requirements.txt",
    region=REGION,
)
print(Path("Dockerfile").read_text())

생성된 Dockerfile 의 마지막 줄 `opentelemetry-instrument python -m agent.runtime_ch02` 에 주목하세요. 06장의 관찰성이 여기서 시작됩니다.

In [ ]:
# CodeBuild 빌드 + 런타임 생성 (약 3~5분)
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)

agent_arn, agent_id = launch_result.agent_arn, launch_result.agent_id
config.save_state(agent_arn=agent_arn, agent_id=agent_id)
print("Agent ARN:", agent_arn)

In [ ]:
wait_for(lambda: agentcore_runtime.status().endpoint["status"],
         ready={"READY"}, failed={"CREATE_FAILED", "UPDATE_FAILED"}, interval=10, label="Runtime DEFAULT 엔드포인트")

## 5. 배포된 에이전트 호출 & 세션 격리 확인 (슬라이드 16)

`InvokeAgentRuntime` API 를 **IAM(SigV4)** 인증으로 호출합니다. (03장에서 JWT 인증으로 바꿉니다.)
`runtimeSessionId` 가 같은 요청은 **같은 microVM** 으로, 다른 세션은 **완전히 격리된 microVM** 으로 라우팅됩니다.

In [ ]:
from demo_helpers.invoke import invoke_with_iam, new_session_id, show

session_a = new_session_id("customer-a")
show(invoke_with_iam(agent_arn, "안녕하세요, 제 이름은 김민수예요. 모니터 보증 기간이 얼마나 되나요?", session_a))

In [ ]:
# 같은 세션 → 같은 microVM → 이전 대화를 기억
show(invoke_with_iam(agent_arn, "제 이름이 뭐였죠?", session_a))

In [ ]:
# 다른 세션 → 새 microVM → 다른 고객의 대화에 접근 불가
session_b = new_session_id("customer-b")
show(invoke_with_iam(agent_arn, "제 이름이 뭐였죠?", session_b))

👉 `microvm_boot_id` 가 세션마다 다릅니다. 로컬 실행과 달리 **세션별 전용 microVM(컴퓨팅+메모리+파일시스템)** 이 할당되어
고객 간 상태가 절대 섞이지 않습니다.

## 6. 세션 수명 주기 (슬라이드 17)
- 세션은 **유휴 15분** 또는 **최대 8시간** 이 지나면 종료되고 microVM 은 폐기됩니다(값은 구성 가능).
- `StopRuntimeSession` 으로 즉시 종료할 수도 있습니다. 종료 후 같은 세션 ID 로 호출하면 **새 microVM** 이 시작됩니다.

In [ ]:
data_client = boto3.client("bedrock-agentcore", region_name=REGION)
data_client.stop_runtime_session(agentRuntimeArn=agent_arn, runtimeSessionId=session_a, qualifier="DEFAULT")
print("세션 A 종료 요청 완료")
time.sleep(5)

show(invoke_with_iam(agent_arn, "제 이름이 뭐였죠?", session_a))

👉 같은 세션 ID 지만 `microvm_boot_id` 가 바뀌었고 대화 내용도 사라졌습니다.
**microVM 의 상태는 휘발성**입니다 → 세션을 넘어 기억하려면 **AgentCore Memory(05장)** 가 필요합니다.

## 7. 버전과 엔드포인트 (슬라이드 13–15)
- 런타임을 생성/업데이트할 때마다 **변경 불가능한 버전**(V1, V2, …)이 생깁니다.
- `DEFAULT` 엔드포인트는 항상 **최신 버전**을 가리킵니다.
- 프로덕션 트래픽용 `PROD` 엔드포인트를 만들어 **현재 버전에 고정**해 두겠습니다.
  이후 챕터에서 에이전트가 계속 업데이트되어도 `PROD` 는 영향을 받지 않으며, 06장에서 검증 후 승격합니다.

In [ ]:
control = boto3.client("bedrock-agentcore-control", region_name=REGION)

current_version = control.get_agent_runtime(agentRuntimeId=agent_id)["agentRuntimeVersion"]
print("현재(최신) 버전:", current_version)

try:
    control.create_agent_runtime_endpoint(
        agentRuntimeId=agent_id, name="PROD", agentRuntimeVersion=current_version,
        description="Production endpoint (pinned)",
    )
    print("PROD 엔드포인트 생성 요청")
except control.exceptions.ConflictException:
    print("PROD 엔드포인트가 이미 존재합니다.")

wait_for(lambda: control.get_agent_runtime_endpoint(agentRuntimeId=agent_id, endpointName="PROD")["status"],
         ready={"READY"}, failed={"CREATE_FAILED", "UPDATE_FAILED"}, label="PROD 엔드포인트")
config.save_state(prod_version=current_version)

In [ ]:
resp = control.list_agent_runtime_endpoints(agentRuntimeId=agent_id)
for ep in resp.get("runtimeEndpoints", []):
    print(f"{ep.get('name'):8} → liveVersion={ep.get('liveVersion')}  targetVersion={ep.get('targetVersion')}  ({ep.get('status')})")
    print(f"           {ep.get('agentRuntimeEndpointArn')}")

In [ ]:
# qualifier 로 호출할 엔드포인트를 선택:  POST /runtimes/{EncodedAgentARN}/invocations?qualifier=PROD
show(invoke_with_iam(agent_arn, "노트북 가격 알려주세요.", new_session_id("prod"), qualifier="PROD"))

## ✅ Chapter 02 정리

| 달성한 것 | 아직 부족한 것 (→ 다음 챕터) |
|---|---|
| 코드 4줄로 Strands 에이전트를 서버리스 배포 | IAM 자격 증명만 있으면 누구나 호출 가능 — **고객 인증 없음** |
| 세션별 microVM 격리, 세션 수명 주기 이해 | 에이전트가 **지금 대화하는 고객이 누구인지 모름** |
| 버전과 `DEFAULT`/`PROD` 엔드포인트 | 외부(배송 파트너) API 를 호출할 **자격 증명 관리 방법 없음** |

➡️ **[03_SecurityIdentity_Demo](03_SecurityIdentity_Demo.ipynb)**: Cognito 로그인 사용자만 에이전트를 호출하도록 하고,
AgentCore Identity 로 외부 API 키를 안전하게 사용합니다.